In [4]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from clean_and_collect_power_data_retry import Clean
from tqdm import tqdm
import pyarrow as pa
from datetime import timedelta, date

In [5]:
for package_choice in [np, pd, pa]:
    print(package_choice.__name__)
    print(package_choice.__version__)

numpy
2.3.5
pandas
3.0.0
pyarrow
23.0.0


In [6]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
test_path = '../../../../data_ds_project/testing_yearly_parquet/'

In [7]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [8]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [9]:
test_path = '../../../../data_ds_project/testing_yearly_parquet/'
# amend as necessary

For testing, just save locally (don't upload it, I think)

In [10]:
test_10 = Clean(10, test_path, systems_cleaned, None, './test_results_10_first/')

In [11]:
my_year = 2016
year_2016_data_basic = test_10.extract_years_data_parquet(my_year)
year_2016_data_standard = test_10.standardize_dataframe(year_2016_data_basic)
year_2016_data_rem = test_10.remove_small_values(year_2016_data_standard)
year_2016_good_days = test_10.keep_good_days_only(year_2016_data_rem)
year_2016_energy_readings = test_10.convert_to_energy_LRS(year_2016_good_days)

	 now beginning extract_years_data_parquet
	beginning standardize_dataframe
                      time  ac_power_kW  year
0      2016-01-01 00:00:00    -0.001732  2016
1      2016-01-01 00:01:00    -0.001776  2016
2      2016-01-01 00:02:00    -0.001721  2016
3      2016-01-01 00:03:00    -0.001743  2016
4      2016-01-01 00:04:00    -0.001687  2016
...                    ...          ...   ...
518820 2016-12-31 23:55:00    -0.001220  2016
518821 2016-12-31 23:56:00    -0.001254  2016
518822 2016-12-31 23:57:00    -0.001209  2016
518823 2016-12-31 23:58:00    -0.001309  2016
518824 2016-12-31 23:59:00    -0.001265  2016

[518825 rows x 3 columns]
                      time     power
0      2016-01-01 00:00:00 -0.001732
1      2016-01-01 00:01:00 -0.001776
2      2016-01-01 00:02:00 -0.001721
3      2016-01-01 00:03:00 -0.001743
4      2016-01-01 00:04:00 -0.001687
...                    ...       ...
518820 2016-12-31 23:55:00 -0.001220
518821 2016-12-31 23:56:00 -0.001254
518822 2016-

In [12]:
test_1200 = Clean(1200, test_path, systems_cleaned, 'inverter', './test_results_1200_i_renew/')

In [13]:
def part_by_part(clean_obj: Clean, year: int):
    try:
        year_data_basic = clean_obj.extract_years_data_parquet(year)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with extract_years_data_parquet')
        raise e
    try:
        year_data_standard = clean_obj.standardize_dataframe(year_data_basic)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with standardize_dataframe')
        raise e
    try:
        year_data_rem = clean_obj.remove_small_values(year_data_standard)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with remove_small_values')
        raise e
    try:
        year_data_good_days = clean_obj.keep_good_days_only(year_data_rem)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with keep_good_days_only (probably good_days).')
        raise e
    try:
        year_data_energy_readings = clean_obj.convert_to_energy_LRS(year_data_good_days)
    except BaseException as e:
        print(f'Trouble with {clean_obj.system_id}, part {clean_obj.meter_or_inverter}, year {year}.')
        print(f'Trouble with convert_to_energy_LRS')
        raise e
    return (year_data_basic, year_data_standard, year_data_rem, year_data_good_days, year_data_energy_readings)

In [15]:
test_1200_year_2012 = part_by_part(test_1200, 2012)

	 now beginning extract_years_data_parquet
	beginning standardize_dataframe
                     time  ac_power_kW_inverter  ac_power_kW_meter  year
0     2012-01-01 07:15:31                   NaN                NaN  2012
1     2012-01-01 07:20:31              0.000000                NaN  2012
2     2012-01-01 07:25:32              0.000210                NaN  2012
3     2012-01-01 07:30:32              0.017430                NaN  2012
4     2012-01-01 07:35:32              0.106918                NaN  2012
...                   ...                   ...                ...   ...
34004 2012-12-13 16:45:32              0.024229                NaN  2012
34005 2012-12-13 16:50:32              0.000000                NaN  2012
34006 2012-12-13 16:55:32              0.000000                NaN  2012
34007 2012-12-13 17:00:32              0.000000                NaN  2012
34008 2012-12-13 17:05:32              0.000000                NaN  2012

[34009 rows x 4 columns]
		entered meter_or_inv

In [16]:
data_1200_inv_bas, data_1200_inv_std, data_1200_inv_rem, data_1200_inv_good, data_1200_year_ener = test_1200_year_2012

In [ ]:
data_1200_inv_bas['ac_power_kW_inverter'].max()

np.float64(48.6627193)

In [ ]:
data_1200_inv_std

,time,power
1,2012-01-01 07:20:31,0.000000
2,2012-01-01 07:25:32,0.000210
3,2012-01-01 07:30:32,0.017430
4,2012-01-01 07:35:32,0.106918
5,2012-01-01 07:40:32,0.203117
...,...,...
34004,2012-12-13 16:45:32,0.024229
34005,2012-12-13 16:50:32,0.000000
34006,2012-12-13 16:55:32,0.000000
34007,2012-12-13 17:00:32,0.000000


In [ ]:
data_1200_inv_rem

,time,power
7,2012-01-01 07:50:32,0.590158
8,2012-01-01 07:55:32,1.115535
9,2012-01-01 08:00:32,1.692868
10,2012-01-01 08:05:32,2.239591
11,2012-01-01 08:10:33,2.888722
...,...,...
33998,2012-12-13 16:15:31,1.936600
33999,2012-12-13 16:20:31,1.618567
34000,2012-12-13 16:25:31,1.291200
34001,2012-12-13 16:30:31,0.940933


In [ ]:
systems_cleaned[systems_cleaned['system_id']==1200]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
11,1200,Distributed Sun - BWI Hilton,"Linthicum Heights, MD",America/New_York,39.1958,-76.6808,155.0,51.84,Cfa,34,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,3365,2011


In [ ]:
data_1200_inv_std

,time,power


Quick loop

In [ ]:
for system_id in tqdm(enough_data_parquet_power_list):
    # grab columns from a blank year
    blank_year_pq = pq.ParquetDataset(
        f'{test_path}{system_id}/',
        filters = [('year', '==', 1990)]
    )
    blank_year_df = blank_year_pq.read().to_pandas()
    pow_cols = [col_name for col_name in blank_year_df.columns if ('pow' in col_name)]
    is_inv = any([('inv' in col_name) for col_name in pow_cols])
    is_met = any([('met' in col_name) for col_name in pow_cols])
    if is_inv and is_met:
        met_inv_inputs = ['inverter', 'meter']
        met_inv_names = met_inv_inputs
    elif is_inv:
        met_inv_inputs = ['inverter',]
        met_inv_names = met_inv_inputs
    elif is_met:
        met_inv_inputs = ['meter',]
        met_inv_names = met_inv_inputs
    else:
        met_inv_inputs = [None,]
        met_inv_names = ['unknown',]
    for j in range(len(met_inv_inputs)):
        met_or_inv = met_inv_inputs[j]
        met_or_inv_name = met_inv_names[j]
        system_cleaning_module = Clean(system_id, test_path, systems_cleaned,
                                       met_or_inv, 
                                       f'./test_results/')
        system_cleaning_module.clean_all_and_write_to_file()


  0%|          | 0/82 [00:00<?, ?it/s]

100%|██████████| 82/82 [02:30<00:00,  1.83s/it]


In [ ]:
good_timezone_systems = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [ ]:
for system_id in good_timezone_systems:
    # grab columns from a blank year
    blank_year_pq = pq.ParquetDataset(
        f'{test_path}{system_id}/',
        filters = [('year', '==', 1990)]
    )
    blank_year_df = blank_year_pq.read().to_pandas()
    pow_cols = [col_name for col_name in blank_year_df.columns if ('pow' in col_name)]
    is_inv = any([('inv' in col_name) for col_name in pow_cols])
    is_met = any([('met' in col_name) for col_name in pow_cols])
    if is_inv and is_met:
        met_inv_inputs = ['inverter', 'meter']
        met_inv_names = met_inv_inputs
    elif is_inv:
        met_inv_inputs = ['inverter',]
        met_inv_names = met_inv_inputs
    elif is_met:
        met_inv_inputs = ['meter',]
        met_inv_names = met_inv_inputs
    else:
        met_inv_inputs = [None,]
        met_inv_names = ['unknown',]
    for j in range(len(met_inv_inputs)):
        met_or_inv = met_inv_inputs[j]
        met_or_inv_name = met_inv_names[j]
        system_cleaning_module = Clean(system_id, test_path, systems_cleaned,
                                       met_or_inv, 
                                       f'./test_results_b/')
        system_cleaning_module.clean_all_and_write_to_file()

Now processing system 4, year 2007.
	 now beginning extract_years_data_parquet
	beginning standardize_dataframe
                     time  ac_power_kW  year
0     2007-08-26 00:00:00    -0.000006  2007
1     2007-08-26 00:15:00    -0.000029  2007
2     2007-08-26 00:30:00    -0.000029  2007
3     2007-08-26 00:45:00    -0.000020  2007
4     2007-08-26 01:00:00    -0.000021  2007
...                   ...          ...   ...
10749 2007-12-31 22:45:00    -0.000095  2007
10750 2007-12-31 23:00:00    -0.000095  2007
10751 2007-12-31 23:15:00    -0.000095  2007
10752 2007-12-31 23:30:00    -0.000095  2007
10753 2007-12-31 23:45:00    -0.000095  2007

[10754 rows x 3 columns]
                     time     power
0     2007-08-26 00:00:00 -0.000006
1     2007-08-26 00:15:00 -0.000029
2     2007-08-26 00:30:00 -0.000029
3     2007-08-26 00:45:00 -0.000020
4     2007-08-26 01:00:00 -0.000021
...                   ...       ...
10749 2007-12-31 22:45:00 -0.000095
10750 2007-12-31 23:00:00 -0.00009

In [ ]:
def good_runs(my_dates):
    runs_list = []
    num_dates = len(my_dates)
    if num_dates == 0:
        return []
    elif num_dates == 1:
        return [my_dates,]
    else:
        dates_ser = pd.Series(
            data = my_dates,
            name = 'date',
        )
        dates_ser = dates_ser.sort_values()
        dates_df = pd.DataFrame(dates_ser)
        dates_df.loc[:, 'forward_diff'] = -dates_df['date'].diff(periods=-1)
        dates_df.loc[:, 'good_forward'] = dates_df.apply(
            lambda row: row['forward_diff'] == timedelta(days=1),
            axis=1
        )
        end_runs = dates_df[~dates_df['good_forward']]
        num_runs = end_runs.shape[0]
        for j in range(num_runs):
            if j == 0:
                start_ind = 0
            else:
                start_ind = end_runs.index[j-1] + 1
            end_ind = end_runs.index[j]
            jth_run = dates_df.loc[start_ind:end_ind]
            runs_list.append(sorted(jth_run['date']))
        return runs_list

In [ ]:
def good_runs_series(dates_ser):
    runs_list = []
    num_dates = len(dates_ser)
    if num_dates == 0:
        return []
    elif num_dates == 1:
        return [[dates_ser.at[0],],]
    else:
        dates_ser = dates_ser.sort_values()
        dates_df = pd.DataFrame(dates_ser)
        dates_df.loc[:, 'forward_diff'] = -dates_df['date'].diff(periods=-1)
        dates_df.loc[:, 'good_forward'] = dates_df.apply(
            lambda row: row['forward_diff'] == timedelta(days=1),
            axis=1
        )
        end_runs = dates_df[~dates_df['good_forward']]
        num_runs = end_runs.shape[0]
        for j in range(num_runs):
            if j == 0:
                start_ind = 0
            else:
                start_ind = end_runs.index[j-1] + 1
            end_ind = end_runs.index[j]
            jth_run = dates_df.loc[start_ind:end_ind]
            runs_list.append(sorted(jth_run['date']))
        return runs_list

In [ ]:
test_read = pd.read_csv('./test_results_b/good_days/4_good_days_None.csv')

In [ ]:
type(test_read)

pandas.DataFrame

In [ ]:
test_read = test_read['date']

In [ ]:
test_read.head()

0    2007-09-01
1    2007-09-03
2    2007-09-04
3    2007-09-07
4    2007-09-08
Name: date, dtype: str

In [ ]:
test_read = test_read.astype('datetime[s]')

TypeError: data type 'datetime[s]' not understood

In [ ]:
test_read

0      2007-09-01
1      2007-09-03
2      2007-09-04
3      2007-09-07
4      2007-09-08
          ...    
2887   2023-02-20
2888   2023-02-21
2889   2023-02-26
2890   2023-02-27
2891   2023-02-28
Name: date, Length: 2892, dtype: datetime64[s]

In [ ]:
test_read.dtype

<StringDtype(na_value=nan)>

In [ ]:
good_runs_series(test_read)

[[Timestamp('2007-09-01 00:00:00')],
 [Timestamp('2007-09-03 00:00:00'), Timestamp('2007-09-04 00:00:00')],
 [Timestamp('2007-09-07 00:00:00'), Timestamp('2007-09-08 00:00:00')],
 [Timestamp('2007-09-11 00:00:00'),
  Timestamp('2007-09-12 00:00:00'),
  Timestamp('2007-09-13 00:00:00'),
  Timestamp('2007-09-14 00:00:00'),
  Timestamp('2007-09-15 00:00:00')],
 [Timestamp('2007-09-18 00:00:00'), Timestamp('2007-09-19 00:00:00')],
 [Timestamp('2007-09-21 00:00:00'),
  Timestamp('2007-09-22 00:00:00'),
  Timestamp('2007-09-23 00:00:00')],
 [Timestamp('2007-09-25 00:00:00'),
  Timestamp('2007-09-26 00:00:00'),
  Timestamp('2007-09-27 00:00:00'),
  Timestamp('2007-09-28 00:00:00'),
  Timestamp('2007-09-29 00:00:00'),
  Timestamp('2007-09-30 00:00:00'),
  Timestamp('2007-10-01 00:00:00'),
  Timestamp('2007-10-02 00:00:00'),
  Timestamp('2007-10-03 00:00:00'),
  Timestamp('2007-10-04 00:00:00'),
  Timestamp('2007-10-05 00:00:00'),
  Timestamp('2007-10-06 00:00:00')],
 [Timestamp('2007-10-08 00:

In [ ]:
run_lengths = []
for system_id in good_timezone_systems:
    print(f'System {system_id}')
    # see runs of good days
    for met_or_inv in ('None', 'inverter', 'meter'):
        try:
            my_good_days = pd.read_csv(f'./test_results_b/good_days/{system_id}_good_days_{met_or_inv}.csv')
        except FileNotFoundError:
            continue
        except BaseException as e:
            raise e
        my_good_days = my_good_days['date']
        my_good_days = my_good_days.astype('datetime64[s]')
        print(met_or_inv)
        for good_run in good_runs_series(my_good_days):
            run_lengths.append(len(good_run))
run_lengths_df = pd.DataFrame(
    data=run_lengths,
    columns=['run_length']
)
run_lengths_df.value_counts().sort_index()

System 4
None
System 10
None
System 33
None
System 36
None
System 50
None
System 51
None
System 1199
inverter
System 1204
inverter
System 1283
inverter
meter
System 1284
None
System 1289
None
System 1332
inverter
meter
System 4902
inverter
meter
System 4903
inverter
meter


run_length
1             3807
2             1930
3             1147
4              720
5              513
6              347
7              209
8              204
9              133
10             102
11              59
12              39
13              36
14              28
15              22
16              24
17              11
18              12
19               7
20               8
21               3
22               6
23               1
24               2
25               2
26               2
30               1
31               1
33               1
36               1
91               1
Name: count, dtype: int64

In [ ]:
my_counts = run_lengths_df.value_counts().sort_index()

In [ ]:
my_counts.iloc[-19:]

run_length
13            36
14            28
15            22
16            24
17            11
18            12
19             7
20             8
21             3
22             6
23             1
24             2
25             2
26             2
30             1
31             1
33             1
36             1
91             1
Name: count, dtype: int64